# Lesson 5 Demo — Working with Hugging Face in Jupyter (run in Colab)

In this short demo, we will use the Hugging Face ecosystem from Python.

The goal is not to train a model from scratch.  
The goal is to understand how a developer can:

1. Load a model from Hugging Face.
2. Run inference using a simple pipeline.
3. Compare outputs from different models.
4. Build a tiny evaluation dataset.
5. Apply a simple CI/CD-style quality gate.

This connects directly to the previous lesson on optimization and CI/CD for LLM applications.

## What you should learn

By the end of this notebook, you should understand that Hugging Face is not only a website for browsing models.

It is also a developer ecosystem that provides:

- model discovery through the Model Hub
- reusable model interfaces through `transformers`
- datasets for experimentation and evaluation
- simple inference abstractions through `pipeline()`

The key idea is this:

> You do not usually interact with raw model weights directly.  
> You interact with models through libraries, APIs, pipelines, and evaluation workflows.

## 1. Install and import the required packages

We will use:

- `transformers` for Hugging Face model pipelines
- `torch` as the deep learning backend
- `pandas` to display outputs clearly in tables

If you already have these packages installed, the install cell may finish quickly.

In [ ]:
#!pip install -q transformers torch pandas

In [ ]:
from transformers import pipeline
import pandas as pd

# This part disables ssl verification, do not run these in a production code, but these lines make the process more portable for a demo
import os
import certifi

cert_path = certifi.where()

os.environ["SSL_CERT_FILE"] = cert_path
os.environ["REQUESTS_CA_BUNDLE"] = cert_path
os.environ["CURL_CA_BUNDLE"] = cert_path

print(cert_path)

## 2. First Hugging Face pipeline: sentiment analysis

The `pipeline()` function is one of the simplest ways to use Hugging Face models.

It hides several low-level steps:

1. loading the model
2. loading the tokenizer
3. converting text into tokens
4. running inference
5. converting raw outputs into readable labels

We will start with a basic sentiment analysis model.

In [ ]:
from transformers import pipeline

sentiment = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english"
)

texts = [
    "The Hugging Face ecosystem makes it easy to experiment with models.",
    "This model is too slow and the result is not useful.",
    "The answer is technically correct, but not very helpful."
]

results = sentiment(texts)
results

## 3. Put model outputs into a table

In real AI applications, model outputs should not just be printed and forgotten.

They often need to be:

- logged
- inspected
- evaluated
- compared
- monitored over time

Here we convert the model outputs into a small table.

In [ ]:
df = pd.DataFrame({
    "text": texts,
    "label": [r["label"] for r in results],
    "score": [r["score"] for r in results]
})

df

## 4. Compare two models on the same inputs

One of the most important lessons in AI application development is that different models behave differently.

Even if two models support the same task, they may produce different labels, confidence scores, or edge-case behavior.

This is why model selection should be treated as an engineering decision, not guesswork.

In [ ]:
model_a = pipeline("sentiment-analysis")

model_b = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest"
)

outputs_a = model_a(texts)
outputs_b = model_b(texts)

comparison = pd.DataFrame({
    "text": texts,
    "default_model_label": [r["label"] for r in outputs_a],
    "default_model_score": [r["score"] for r in outputs_a],
    "roberta_label": [r["label"] for r in outputs_b],
    "roberta_score": [r["score"] for r in outputs_b],
})

comparison

## Discussion

Look at the comparison table.

Ask:

- Do the two models agree?
- Do their confidence scores differ?
- Which model would you trust more for this task?
- Is popularity enough to decide?
- Would these results generalize to your own application?

This is the practical reason why we evaluate models using our own data.

## 5. Create a tiny golden dataset

In the previous lesson, we discussed golden datasets.

A golden dataset is a curated set of examples used to evaluate whether a model behaves acceptably.

In production, this dataset would be much larger and more carefully designed.

For a classroom demo, we will use a tiny dataset with expected labels.

In [ ]:
golden = pd.DataFrame([
    {
        "text": "The product works perfectly and saves me time.",
        "expected": "POSITIVE"
    },
    {
        "text": "The app crashes every time I open it.",
        "expected": "NEGATIVE"
    },
    {
        "text": "The documentation exists, but it is hard to follow.",
        "expected": "NEGATIVE"
    },
    {
        "text": "The new interface is clean, fast, and easy to use.",
        "expected": "POSITIVE"
    },
    {
        "text": "The installation process failed and the error message was confusing.",
        "expected": "NEGATIVE"
    },
])

golden

## 6. Evaluate a model against the golden dataset

Now we run the model on the examples in our golden dataset.

Then we compare the predicted label to the expected label.

This is a simplified version of an evaluation workflow.

In [ ]:
predictions = sentiment(golden["text"].tolist())

golden_eval = golden.copy()
golden_eval["predicted"] = [p["label"] for p in predictions]
golden_eval["score"] = [p["score"] for p in predictions]
golden_eval["correct"] = golden_eval["predicted"] == golden_eval["expected"]

golden_eval

In [ ]:
accuracy = golden_eval["correct"].mean()
print(f"Accuracy on golden dataset: {accuracy:.2%}")

## 7. Add a simple CI/CD-style quality gate

In a real CI/CD pipeline, we would not manually inspect every result.

Instead, we define a threshold.

For example:

> If accuracy is below 80%, block the change.

This is a simplified version of an automated quality gate.

In [ ]:
MIN_ACCURACY = 0.80

if accuracy >= MIN_ACCURACY:
    print("PASS: model is acceptable for this small evaluation set")
else:
    print("FAIL: model needs more testing, a better model, or better task design")

## 8. Optional: evaluate the second model

Now we repeat the same evaluation using the second model.

This shows how the same golden dataset can be reused to compare candidate models.

In [ ]:
predictions_b = model_b(golden["text"].tolist())

golden_eval_b = golden.copy()
golden_eval_b["predicted"] = [p["label"] for p in predictions_b]
golden_eval_b["score"] = [p["score"] for p in predictions_b]
golden_eval_b["correct"] = golden_eval_b["predicted"] == golden_eval_b["expected"]

golden_eval_b

In [ ]:
accuracy_b = golden_eval_b["correct"].mean()

summary = pd.DataFrame([
    {"model": "default sentiment model", "accuracy": accuracy},
    {"model": "twitter-roberta sentiment model", "accuracy": accuracy_b},
])

summary

## 9. What this demo showed

This notebook demonstrated a small but important workflow:

1. Load models from Hugging Face.
2. Run inference using pipelines.
3. Compare models on the same inputs.
4. Create a tiny golden dataset.
5. Measure model performance.
6. Apply a CI/CD-style quality gate.

The important lesson is not the sentiment task itself.

The important lesson is the workflow:

> Model selection should be based on evaluation, not assumptions.

## 10. How this scales in real projects

In real AI applications, this same structure becomes more sophisticated.

Instead of a tiny dataset, you may have hundreds or thousands of examples.

Instead of simple accuracy, you may measure:

- answer quality
- hallucination rate
- safety compliance
- latency
- cost
- consistency across runs

Instead of manually running the notebook, the evaluation may run automatically in a CI/CD pipeline.

The same basic pattern still applies:

> Change → Run model → Evaluate → Measure → Compare → Decide

In [ ]:
!pip install --upgrade pip